# 📈 Stock Price Prediction & Forecasting
## Data Analyst Portfolio Project
---
**Author:** Your Name  
**Dataset:** Yahoo Finance (via yfinance)  
**Models:** ARIMA · Facebook Prophet  
**Stock:** TCS (Tata Consultancy Services)

## 1. 📥 Import Libraries & Download Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ Libraries imported successfully!')

In [ ]:
# Download TCS stock data (change ticker to try other stocks)
TICKER     = 'TCS.NS'
START_DATE = '2019-01-01'
END_DATE   = '2024-12-31'

df = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)
df.reset_index(inplace=True)
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df.dropna(inplace=True)

print(f'✅ Downloaded {len(df)} rows for {TICKER}')
print(f'   Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')
df.head()

## 2. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('📊 Basic Statistics:')
df[['Open','High','Low','Close','Volume']].describe().round(2)

In [ ]:
# Check missing values
print('🔍 Missing Values:')
print(df.isnull().sum())

In [ ]:
# Plot closing price history
plt.figure(figsize=(14, 5))
plt.plot(df['Date'], df['Close'], linewidth=1.8, color='steelblue')
plt.title(f'{TICKER} — Historical Closing Price', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (₹)')
plt.tight_layout()
plt.show()

In [ ]:
# Moving averages
df['MA_20'] = df['Close'].rolling(20).mean()
df['MA_50'] = df['Close'].rolling(50).mean()

plt.figure(figsize=(14, 5))
plt.plot(df['Date'], df['Close'], label='Close Price', linewidth=1.5, alpha=0.8)
plt.plot(df['Date'], df['MA_20'],  label='MA 20',  linewidth=2, linestyle='--')
plt.plot(df['Date'], df['MA_50'],  label='MA 50',  linewidth=2, linestyle=':')
plt.title(f'{TICKER} — Price with Moving Averages', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (₹)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Daily returns distribution
df['Daily_Return'] = df['Close'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['Daily_Return'].plot(ax=axes[0], title='Daily Returns (%)', linewidth=1, color='steelblue')
df['Daily_Return'].dropna().plot.hist(ax=axes[1], bins=50, color='steelblue', edgecolor='white',
                                       title='Daily Returns Distribution')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
plt.tight_layout()
plt.show()

print(f'Mean daily return : {df["Daily_Return"].mean():.4f}%')
print(f'Std daily return  : {df["Daily_Return"].std():.4f}%')

In [ ]:
# Volume chart
plt.figure(figsize=(14, 4))
plt.bar(df['Date'], df['Volume'], color='steelblue', alpha=0.7, width=2)
plt.title(f'{TICKER} — Trading Volume', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.tight_layout()
plt.show()

## 3. 📊 Stationarity Check (ADF Test)

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    print(f'\n📊 ADF Test — {name}')
    print(f'   ADF Statistic : {result[0]:.4f}')
    print(f'   p-value       : {result[1]:.4f}')
    if result[1] < 0.05:
        print(f'   ✅ STATIONARY  (p < 0.05 — good for ARIMA)')
    else:
        print(f'   ⚠️  NOT STATIONARY — differencing needed')

close = df['Close'].astype(float)

adf_test(close, 'Closing Price (Original)')
adf_test(close.diff(), 'Closing Price (1st Difference)')

## 4. 🔮 ARIMA Model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

# Train / test split (80/20)
close_series = close.reset_index(drop=True)
close_series.index = pd.to_datetime(df['Date'].values)

split = int(len(close_series) * 0.80)
train = close_series[:split]
test  = close_series[split:]

print(f'Train size: {len(train)} | Test size: {len(test)}')

In [ ]:
# Rolling one-step ARIMA predictions
ORDER   = (5, 1, 0)
history = list(train)
preds   = []

print('Fitting ARIMA... (this may take ~1 minute)')
for i, obs in enumerate(test):
    model = ARIMA(history, order=ORDER).fit()
    yhat  = model.forecast(steps=1)[0]
    preds.append(yhat)
    history.append(obs)
    if (i+1) % 50 == 0:
        print(f'  Step {i+1}/{len(test)} done')

preds  = np.array(preds)
actual = test.values

rmse = math.sqrt(mean_squared_error(actual, preds))
mae  = mean_absolute_error(actual, preds)
mape = np.mean(np.abs((actual - preds) / actual)) * 100

print(f'\n📈 ARIMA Performance:')
print(f'   RMSE : {rmse:.2f}')
print(f'   MAE  : {mae:.2f}')
print(f'   MAPE : {mape:.2f}%')

In [ ]:
# Plot ARIMA predictions
plt.figure(figsize=(14, 5))
plt.plot(test.index, actual, label='Actual',    linewidth=2)
plt.plot(test.index, preds,  label='ARIMA Pred', linestyle='--', linewidth=1.8)
plt.title(f'ARIMA{ORDER} — Actual vs Predicted (Test Set)', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (₹)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 30-day future forecast
FORECAST_DAYS  = 30
final_model    = ARIMA(close_series, order=ORDER).fit()
arima_forecast = final_model.forecast(steps=FORECAST_DAYS)
last_date      = close_series.index[-1]
future_dates   = pd.bdate_range(start=last_date, periods=FORECAST_DAYS + 1)[1:]

plt.figure(figsize=(14, 5))
plt.plot(close_series[-90:], label='Historical (last 90 days)', linewidth=2)
plt.plot(future_dates, arima_forecast.values, label=f'{FORECAST_DAYS}-day Forecast',
         color='orange', linestyle='--', linewidth=2)
plt.title(f'ARIMA — {FORECAST_DAYS}-Day Price Forecast for {TICKER}', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (₹)')
plt.legend()
plt.tight_layout()
plt.show()

forecast_df = pd.DataFrame({'Date': future_dates, 'ARIMA_Forecast': arima_forecast.values})
print(forecast_df.head(10).to_string(index=False))

## 5. 🔮 Facebook Prophet Model

In [ ]:
from prophet import Prophet

# Prepare Prophet format
prophet_df = df[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})
prophet_df['y'] = prophet_df['y'].astype(float)
prophet_df.head()

In [ ]:
# Train/test split
split_p   = int(len(prophet_df) * 0.80)
train_p   = prophet_df[:split_p]
test_p    = prophet_df[split_p:]

# Fit Prophet
model_p = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
)
model_p.fit(train_p)
print('✅ Prophet model fitted!')

In [ ]:
# Evaluate on test set
future_test  = model_p.make_future_dataframe(periods=len(test_p), freq='B')
fc_all       = model_p.predict(future_test)
test_fc      = fc_all[fc_all['ds'].isin(test_p['ds'])]['yhat'].values
actual_p     = test_p['y'].values[:len(test_fc)]

p_rmse = math.sqrt(mean_squared_error(actual_p, test_fc))
p_mae  = mean_absolute_error(actual_p, test_fc)
p_mape = np.mean(np.abs((actual_p - test_fc) / actual_p)) * 100

print(f'📈 Prophet Performance:')
print(f'   RMSE : {p_rmse:.2f}')
print(f'   MAE  : {p_mae:.2f}')
print(f'   MAPE : {p_mape:.2f}%')

In [ ]:
# Full data forecast
model_full = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
)
model_full.fit(prophet_df)

future   = model_full.make_future_dataframe(periods=FORECAST_DAYS, freq='B')
forecast = model_full.predict(future)
fc_tail  = forecast.tail(FORECAST_DAYS)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
hist_tail = prophet_df.tail(90)
ax.plot(hist_tail['ds'], hist_tail['y'], label='Historical', linewidth=2)
ax.plot(fc_tail['ds'], fc_tail['yhat'], color='green', linestyle='--',
        linewidth=2, label=f'{FORECAST_DAYS}-day Forecast')
ax.fill_between(fc_tail['ds'], fc_tail['yhat_lower'], fc_tail['yhat_upper'],
                alpha=0.25, color='green', label='95% CI')
ax.set_title(f'Prophet — {FORECAST_DAYS}-Day Price Forecast for {TICKER}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (₹)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Prophet components — trend + seasonality
fig = model_full.plot_components(forecast)
plt.tight_layout()
plt.show()

## 6. 📊 Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['ARIMA (5,1,0)', 'Facebook Prophet'],
    'RMSE':  [round(rmse, 2),   round(p_rmse, 2)],
    'MAE':   [round(mae, 2),    round(p_mae, 2)],
    'MAPE':  [f'{mape:.2f}%',   f'{p_mape:.2f}%'],
})

print('\n📊 Model Performance Comparison:')
print(comparison.to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['RMSE', 'MAE']
colors  = ['steelblue', 'seagreen']

for i, metric in enumerate(['RMSE', 'MAE']):
    vals = [rmse if metric == 'RMSE' else mae,
            p_rmse if metric == 'RMSE' else p_mae]
    axes[i].bar(['ARIMA', 'Prophet'], vals, color=colors)
    axes[i].set_title(metric, fontweight='bold')
    axes[i].set_ylabel('Value')

axes[2].bar(['ARIMA', 'Prophet'], [mape, p_mape], color=colors)
axes[2].set_title('MAPE (%)', fontweight='bold')
axes[2].set_ylabel('%')

plt.suptitle('Model Comparison — ARIMA vs Prophet', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. ✅ Conclusion

- **ARIMA** captures linear trends well using autoregressive components
- **Prophet** handles seasonality and holiday effects better, giving lower MAPE
- For short-term forecasting, both models produce similar results
- **Prophet** is preferred for real-world applications due to ease of use and confidence intervals

---
*Next steps: Add sentiment analysis from news headlines or try LSTM for deep learning approach*